In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=uUtKkFQqPRk8T5JB2LXK42l3VOsiFQ&access_type=offline&code_challenge=xlDtmgZTxoaLmBQWBWVnFbw0NeqhXmyLXnRDr_NPjK0&code_challenge_method=S256


Credentials saved to file: [C:\Users\luong\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Cannot find a quota project to add to ADC. You might receive a "quota exceeded" or "API not enabled" error. Run $ gcloud auth application-default set-quota-project to add a quota project.


In [24]:
import os
from pathlib import Path
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
from huggingface_hub import hf_hub_download
# Lấy ở trên đoạn code chạy !gcloud auth application-default login
credentials_location = r"C:\Users\luong\AppData\Roaming\gcloud\application_default_credentials.json" 

gcs_jar_path = r"C:\Users\luong\Documents\GitHub\BTL_BigData\load_rawData\gcs-connector-hadoop3-latest.jar"

if not Path(credentials_location).exists():
    raise FileNotFoundError(f"Credentials file not found: {credentials_location}")

if not Path(gcs_jar_path).exists():
    raise FileNotFoundError(f"GCS connector jar not found: {gcs_jar_path}")

# Resource tuning for local Spark (change these numbers if needed).
driver_memory_gb = 8
executor_memory_gb = 8
reserve_cores = 1

total_cores = os.cpu_count() or 4
spark_cores = max(1, total_cores - reserve_cores)
default_parallelism = max(8, spark_cores * 2)
shuffle_partitions = max(16, spark_cores * 4)
jar_classpath = gcs_jar_path

# Recreate Spark context so new config is applied on rerun.
active_sc = SparkContext._active_spark_context
if active_sc is not None:
    active_sc.stop()

conf = (
    SparkConf()
        .setMaster(f"local[{spark_cores}]")
        .setAppName("test")
        .set("spark.driver.memory", f"{driver_memory_gb}g")
        .set("spark.executor.memory", f"{executor_memory_gb}g")
        .set("spark.default.parallelism", str(default_parallelism))
        .set("spark.sql.shuffle.partitions", str(shuffle_partitions))
        # Use classpath instead of spark.jars to avoid Windows winutils/chmod issue.
        .set("spark.driver.extraClassPath", jar_classpath)
        .set("spark.executor.extraClassPath", jar_classpath)
        .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)
)

sc = SparkContext(conf=conf)
hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)

spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()
spark.conf.set("spark.sql.caseSensitive", "true")
print(f"Spark cores: {spark_cores}/{total_cores}")
print(f"Driver memory: {driver_memory_gb}g, Executor memory: {executor_memory_gb}g")
print(f"defaultParallelism={default_parallelism}, shufflePartitions={shuffle_partitions}")

Spark cores: 15/16
Driver memory: 8g, Executor memory: 8g
defaultParallelism=30, shufflePartitions=60


In [ ]:
def _estimate_output_files_from_json_size(
    json_file_path,
    target_parquet_file_mb=128,
    estimated_compression_ratio=0.30,
    min_files=1,
    max_files=200,
):
    """
    Estimate output parquet file count from source JSON size.

    estimated_parquet_size ~= json_size * estimated_compression_ratio
    output_files ~= estimated_parquet_size / target_parquet_file_mb
    """
    import math

    json_size_bytes = os.path.getsize(json_file_path)
    target_bytes = target_parquet_file_mb * 1024 * 1024

    estimated_parquet_bytes = max(1, int(json_size_bytes * estimated_compression_ratio))
    estimated_files = math.ceil(estimated_parquet_bytes / target_bytes)

    output_files = max(min_files, min(max_files, estimated_files))
    return output_files, json_size_bytes, estimated_parquet_bytes


def upload_meta_data(
    category,
    target_parquet_file_mb=128,
    estimated_compression_ratio=0.30,
    max_output_files=200,
    compression_codec="snappy",
):
    repo_id = "McAuley-Lab/Amazon-Reviews-2023"
    file_in_repo = f"raw/meta_categories/meta_{category}.jsonl"
    bronze_output_path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/{category}"

    print(f"Uploading metadata for category: {category}")

    local_file_path = hf_hub_download(
        repo_id=repo_id,
        filename=file_in_repo,
        repo_type="dataset",
    )

    num_files, json_size_bytes, estimated_parquet_bytes = _estimate_output_files_from_json_size(
        json_file_path=local_file_path,
        target_parquet_file_mb=target_parquet_file_mb,
        estimated_compression_ratio=estimated_compression_ratio,
        min_files=1,
        max_files=max_output_files,
    )

    json_size_gb = json_size_bytes / (1024 ** 3)
    estimated_parquet_gb = estimated_parquet_bytes / (1024 ** 3)

    print(f"Source JSON size: {json_size_gb:.2f} GB")
    print(f"Estimated parquet size: {estimated_parquet_gb:.2f} GB")
    print(f"Target parquet file size: {target_parquet_file_mb} MB")
    print(f"Calculated output files: {num_files}")
    print(f"Compression codec: {compression_codec}")

    df = spark.read.option("multiLine", "false").json(local_file_path)

    current_partitions = df.rdd.getNumPartitions()
    if num_files >= current_partitions:
        df_to_write = df.repartition(num_files)
        partition_action = "repartition"
    else:
        df_to_write = df.coalesce(num_files)
        partition_action = "coalesce"

    print(f"Input partitions: {current_partitions}, using {partition_action} -> {num_files}")

    (
        df_to_write
        .write
        .mode("overwrite")
        .option("compression", compression_codec)
        .parquet(bronze_output_path)
    )

    print(f"Finished writing parquet to: {bronze_output_path}")


In [34]:
upload_meta_data("All_Beauty")

Uploading metadata for category: All_Beauty
Source JSON size: 0.20 GB
Estimated parquet size: 0.06 GB
Target parquet file size: 128 MB
Calculated output files: 1
Input partitions: 30, using coalesce -> 1
Finished writing parquet to: gs://amazon-reviews-lakehouse-data/bronze-zone/meta-data/All_Beauty
